# Stage 0 - run the error-spectrum gate on Kaggle

Clones the public `Leviathan2006/benchmark` repo, installs it, runs the
estimator tests, then two smoke measurements (synthetic + `diff_ks`).

**Before running:** turn **Add-ons -> Internet -> On**.

This does NOT launch the full sweep - smoke configs only.

In [ ]:
# --- clone -------------------------------------------------------------------
import os, subprocess, pathlib

REPO_DIR = "/kaggle/working/benchmark"
BRANCH = "stage0-error-spectrum"   # branch the Stage 0 files live on; set to "main" after merge
URL = "https://github.com/Leviathan2006/benchmark.git"

if pathlib.Path(REPO_DIR).exists():
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("HEAD:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("stage0 present:", pathlib.Path("src/stage0_error_spectrum.py").exists())

In [ ]:
# --- install ---------------------------------------------------------------
# --no-deps: skip the jax/trainax/apebench pins in pyproject; add apebench itself.
!pip -q install --no-deps -e /kaggle/working/benchmark
!pip -q install apebench
import rollout_error.stage0_error_spectrum as s0
print("import OK ->", s0.__file__)

In [ ]:
# --- estimator tests (numpy only, no apebench needed) ---------------------
!cd /kaggle/working/benchmark && python -m pytest tests/test_stage0_spectrum.py -q

In [ ]:
# --- smoke 1: synthetic system, known planted exponent (no apebench) -----
!cd /kaggle/working/benchmark && python -m src.stage0_error_spectrum \
    --synthetic --smoke --out /kaggle/working/results

In [ ]:
# --- smoke 2: real APEBench emulator, diff_ks, tiny config ---------------
# If this dies inside extract_emulator / get_test_trajectories, copy the
# structural dump it prints - the apebench API shape is the one thing that
# could not be verified offline.
!cd /kaggle/working/benchmark && python -m src.stage0_error_spectrum \
    --smoke --scenarios diff_ks --out /kaggle/working/results

In [ ]:
# --- show the figures + the parquet -------------------------------------
import glob
import pandas as pd
from IPython.display import Image, display

for png in sorted(glob.glob("/kaggle/working/results/*.png")):
    print(png)
    display(Image(filename=png))

for pq in sorted(glob.glob("/kaggle/working/results/*.parquet")):
    print("\n", pq)
    df = pd.read_parquet(pq)
    cols = [c for c in ["scenario", "k", "flatness", "beta", "beta_se_boot",
                         "beta_se_ols", "rel_error_power", "pearson_corr", "saturated"]
            if c in df.columns]
    display(df[cols].drop_duplicates(subset=["scenario", "k"]).reset_index(drop=True))